# Phase 2 - Difficulty-Aware Inference Scaling

Este notebook analisa quando Best-of-N realmente agrega valor. A pergunta central e custo marginal: quais tarefas compram acuracia com mais inferencia e quais continuam dificeis?


## Objetivo

Antes de Activation Steering, precisamos medir dificuldade, diversidade, temperatura, custo e fronteira de Pareto. Se a amostra for pequena, trate tudo como piloto.


Esta celula localiza o repositorio e permite abrir o notebook tanto da raiz quanto da pasta `notebooks/`.


In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))


Imports analiticos. Eles ficam em modulos do projeto para manter o notebook pequeno e reutilizavel.


In [2]:
import pandas as pd
import matplotlib.pyplot as plt

from slm_steering.analysis import (
    load_runs_from_manifest, discover_run_bundles,
    comparison_frame, attempts_frame,
)
from slm_steering.best_of_k import best_of_k_marginal_curve


Imports da Fase 2: dificuldade, Pareto, diversidade e graficos.


In [3]:
from slm_steering.difficulty import difficulty_frame
from slm_steering.pareto import pareto_frame
from slm_steering.analysis import diversity_frame
from slm_steering.visualization import (
    setup_style, plot_marginal_gain,
    plot_difficulty_distribution, plot_pareto_front,
    plot_diversity_vs_success,
)
setup_style()


A Fase 2 espera resultados em `runs/phase2/`. O manifest e criado pelo script de matriz, mas o notebook tambem consegue descobrir summaries soltos.


In [4]:
PHASE2_DIR = ROOT / "runs" / "phase2"
MANIFEST = PHASE2_DIR / "manifest.json"

PHASE2_DIR, MANIFEST.exists()


(WindowsPath('c:/Users/joaof/slm-inference-steering/runs/phase2'), False)

Use `--dry-run` para revisar a matriz antes de gastar GPU. O comando maior pode demorar bastante dependendo de N e do numero de modelos.


In [5]:
print("python scripts/run_experiment_matrix.py --limit 10 --dry-run")
print("python scripts/run_experiment_matrix.py --limit 10 --skip-existing")
print("python scripts/run_experiment_matrix.py --limit 10 --skip-existing --local-files-only")


python scripts/run_experiment_matrix.py --limit 10 --dry-run
python scripts/run_experiment_matrix.py --limit 10 --skip-existing
python scripts/run_experiment_matrix.py --limit 10 --skip-existing --local-files-only


Carregamos as runs concluida. Se nao houver resultados ainda, as proximas celulas ficam vazias sem quebrar a apresentacao.


In [6]:
if MANIFEST.exists():
    phase2_runs = load_runs_from_manifest(MANIFEST)
else:
    phase2_runs = discover_run_bundles(PHASE2_DIR)

len(phase2_runs)


0

A tabela comparativa resume cada run: modelo, temperatura, N, acuracia e custo. Ela e o ponto de partida para a narrativa experimental.


In [7]:
comparison = comparison_frame(phase2_runs)
cols = ["run", "model_id", "temperature", "requested_n", "early_stop", "pass_at_1", "best_of_n", "generated_tokens"]
comparison[cols] if not comparison.empty else comparison


""


Se poucas tarefas foram avaliadas, marque explicitamente como piloto. Isso evita conclusoes fortes a partir de uma amostra pequena.


In [8]:
if comparison.empty:
    print("Nenhuma run carregada ainda.")
elif comparison["tasks"].max() < 30:
    print("Amostra piloto: interprete padroes como hipoteses, nao conclusoes.")
else:
    print("Amostra mais robusta para comparacoes agregadas.")


Nenhuma run carregada ainda.


Selecionamos uma run para inspecao detalhada. Por padrao, pegamos a primeira; voce pode trocar o indice para montar slides especificos.


In [9]:
selected_run = phase2_runs[0] if phase2_runs else None
selected_run.label if selected_run else None


A curva Best-of-K mostra a acuracia acumulada quando aumentamos o orcamento de amostras.


In [10]:
curve = best_of_k_marginal_curve(selected_run.records) if selected_run else pd.DataFrame()
curve


""


O ganho marginal responde a pergunta economica: a tentativa K ainda compra acuracia ou so aumenta custo?


In [11]:
if not curve.empty:
    plot_marginal_gain(curve)
    plt.show()


Agora classificamos cada tarefa: easy, sampling_sensitive, fragile ou hard. A coluna `solved_by_sampling` destaca onde Best-of-N foi essencial.


In [12]:
difficulty = pd.concat(
    [difficulty_frame(run.records, run_label=run.label) for run in phase2_runs],
    ignore_index=True,
) if phase2_runs else pd.DataFrame()

difficulty.head()


""


A distribuicao de dificuldade mostra se o experimento e trivial ou se ha tarefas que realmente exigem inference scaling.


In [13]:
if not difficulty.empty:
    plot_difficulty_distribution(difficulty)
    plt.show()


Tarefas resolvidas apenas por sampling sao especialmente importantes para o TCC: elas mostram onde Best-of-N compra algo que pass@1 nao entrega.


In [14]:
if difficulty.empty:
    sampling_only = pd.DataFrame()
else:
    sampling_only = difficulty[difficulty["solved_by_sampling"]]

sampling_only[["run", "task_id", "first_success_attempt", "success_rate"]] if not sampling_only.empty else sampling_only


""


Tarefas hard permanecem sem solucao no orcamento usado. Elas sao candidatas naturais para estudar steering, prompt changes ou modelos maiores.


In [15]:
if difficulty.empty:
    hard_tasks = pd.DataFrame()
else:
    hard_tasks = difficulty[difficulty["difficulty"] == "hard"]

hard_tasks[["run", "task_id", "attempts", "generated_tokens"]] if not hard_tasks.empty else hard_tasks


""


A analise por tentativa separa custo de sucesso e falha. Isso mostra se falhas longas estao consumindo grande parte do orcamento.


In [16]:
attempts = pd.concat(
    [attempts_frame(run) for run in phase2_runs],
    ignore_index=True,
) if phase2_runs else pd.DataFrame()

attempts.head()


""


Este boxplot compara tokens gerados por tentativas corretas e incorretas. Se falhas sao caras, early stopping e filtros ficam mais relevantes.


In [17]:
if not attempts.empty:
    attempts.boxplot(column="generated_tokens", by="passed")
    plt.suptitle("")
    plt.title("Tokens por sucesso/falha")
    plt.show()


A diversidade entre respostas ajuda a preparar a discussao de steering. Queremos melhorar direcao sem colapsar diversidade util.


In [18]:
diversity = pd.concat(
    [diversity_frame(run) for run in phase2_runs if len(run.records) > 0],
    ignore_index=True,
) if phase2_runs else pd.DataFrame()

diversity.head()


""


Quando houver N > 1, este grafico sugere se maior diversidade de codigo esta associada a maior taxa de solucao.


In [19]:
if not diversity.empty and diversity["mean_pairwise_code_distance"].notna().any():
    plot_diversity_vs_success(diversity.dropna(subset=["mean_pairwise_code_distance"]))
    plt.show()


A fronteira de Pareto combina custo medio efetivo e taxa de resolucao. Pontos contornados sao nao dominados dentro das runs carregadas.


In [20]:
pareto = pareto_frame(phase2_runs)
pareto.head()


""


Este grafico e central para a entrega: Activation Steering so sera convincente se deslocar essa fronteira para cima/esquerda.


In [21]:
if not pareto.empty:
    plot_pareto_front(pareto)
    plt.show()


Comparacao entre Best-of-N completo e early stopping. Ela so aparece quando a matriz inclui os dois modos para a mesma configuracao.


In [22]:
if not pareto.empty:
    keys = ["model_id", "temperature", "n"]
    early_compare = pareto.pivot_table(index=keys, columns="early_stop", values="mean_effective_tokens")
else:
    early_compare = pd.DataFrame()
early_compare


""


## Leituras Para Slides

- Dificuldade por tarefa mostra onde Best-of-N agrega valor real.
- Ganho marginal revela se N maior ainda compra acuracia.
- Pareto custo-acuracia define a linha de base que steering precisa superar.
- Amostras pequenas devem ser apresentadas como piloto, nao como evidencia final.


## Proxima Hipotese

Se uma futura tecnica de Activation Steering aumentar `pass@1` ou reduzir `mean_effective_tokens` mantendo acuracia, ela desloca a fronteira de Pareto. Essa e a ponte natural para a Fase 3.
